In [3]:
# ------------------------------------------------------------
# Re-parse the multi-label columns, which round-trip through CSV as strings
# (e.g. "['Action', 'Adventure']") rather than real Python lists.
# ------------------------------------------------------------
def parse_list_col(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        return []
    try:
        v = ast.literal_eval(x)
        return v if isinstance(v, list) else []
    except (ValueError, SyntaxError):
        return []

LIST_COLS = ['genres', 'studios', 'themes', 'producers', 'demographics']
for col in LIST_COLS:
    if col in df.columns:
        df[col] = df[col].apply(parse_list_col)

assert 'balanced_score' in df.columns, "balanced_score column missing — load the post-feature-engineering working dataset, not the raw cleaned dataset."

d = df.copy()
print(d[LIST_COLS].head(3))
print("\nRows with balanced_score present:", d['balanced_score'].notna().sum())

                                genres       studios                      themes                                          producers demographics
0            [Action, Drama, Suspense]  [Wit Studio]  [Gore, Military, Survival]  [Production I.G, Dentsu, Mainichi Broadcasting...    [Shounen]
1             [Supernatural, Suspense]    [Madhouse]             [Psychological]  [VAP, Nippon Television Network, Shueisha, D.N...    [Shounen]
2  [Action, Adventure, Drama, Fantasy]       [Bones]                  [Military]  [Aniplex, Square Enix, Mainichi Broadcasting S...    [Shounen]

Rows with balanced_score present: 8227


In [4]:
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8227 entries, 0 to 8226
Data columns (total 36 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   mal_id                8227 non-null   int64  
 1   title                 8227 non-null   object 
 2   title_english         5979 non-null   object 
 3   title_japanese        8209 non-null   object 
 4   title_synonym         5026 non-null   object 
 5   type                  8227 non-null   object 
 6   source                8227 non-null   object 
 7   status                8227 non-null   object 
 8   rating                8227 non-null   object 
 9   score                 8227 non-null   float64
 10  scored_by             8227 non-null   float64
 11  rank                  8227 non-null   float64
 12  popularity            8227 non-null   int64  
 13  members               8227 non-null   int64  
 14  favorites             8227 non-null   int64  
 15  season               

In [5]:
d['balanced_score'].describe()

count    8227.000000
mean        0.500061
std         0.260341
min         0.015704
25%         0.287773
50%         0.507904
75%         0.707418
max         0.999781
Name: balanced_score, dtype: float64

## Recommender System Evaluation — Similarity-Score-Based Validation

The content-based hybrid recommender (60% storyline SVD similarity + 40% genre/theme similarity) was never trained against a labelled "correct recommendation" target — it is a similarity-ranking system, not a classifier. As your teacher noted, the right way to evaluate a system like this is **on its similarity scores themselves**, not on classification metrics that do not apply here. This section implements two concrete, quantitative checks that use only the similarity scores the system already produces (no external labels or additional data are needed):

1. **Similarity lift** — is the average similarity of the system's own top-K recommendations meaningfully higher than the average similarity between *random, unrelated* pairs of anime? If the hybrid similarity score carries real signal, top-K recommended pairs should score far above a random baseline; if the system were producing near-random rankings, the two would be close to equal (lift \u2248 1).
2. **Franchise self-retrieval rate** — for titles that have a clear same-franchise sibling elsewhere in the dataset (detected purely from title text, e.g. *"...2nd Season"*, *"...Part 2"*), does the recommender actually surface that sibling in its own top-10 results? This uses a real, verifiable relevance signal (two titles either are or are not the same franchise) without requiring any hand-labelled relevance data.

Together, these give a defensible, quantitative answer to "does this recommender work" using nothing but the similarity scores the model already computes.

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer, normalize
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# Rebuild the hybrid similarity representation on the full, synopsis-complete
# dataset (independent of the classification split above — the recommender is
# not a time-based predictive model, so it is fit on all available titles).
# ------------------------------------------------------------
dt = d[d['synopsis_clean'].notna()].copy().reset_index(drop=True)
print("Titles available for recommendation:", len(dt))

tfidf_r = TfidfVectorizer(max_features=20000, stop_words='english', min_df=2)
X_text_r = tfidf_r.fit_transform(dt['synopsis_clean'])

svd_r = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_svd_r = svd_r.fit_transform(X_text_r)
X_story = normalize(X_svd_r)

mlb_g = MultiLabelBinarizer()
mlb_t = MultiLabelBinarizer()
G = mlb_g.fit_transform(dt['genres'])
T = mlb_t.fit_transform(dt['themes'])
GT = normalize(np.hstack([G, T]).astype(float))

print("Story embedding shape:", X_story.shape, " Genre/theme embedding shape:", GT.shape)

Titles available for recommendation: 8227
Story embedding shape: (8227, 100)  Genre/theme embedding shape: (8227, 88)


In [16]:
W_STORY, W_GENRE = 0.6, 0.4

def hybrid_similarity_row(idx):
    story_sim = cosine_similarity(X_story[idx:idx+1], X_story).flatten()
    gt_sim = cosine_similarity(GT[idx:idx+1], GT).flatten()
    return W_STORY * story_sim + W_GENRE * gt_sim

rng = np.random.default_rng(RANDOM_STATE)
n_titles = len(dt)

# ------------------------------------------------------------
# 1) Similarity lift: top-K recommended similarity vs. random-pair baseline
# ------------------------------------------------------------
SAMPLE_SIZE = 500
K = 10

sample_idx = rng.choice(n_titles, size=min(SAMPLE_SIZE, n_titles), replace=False)

top_k_sims = []
random_k_sims = []

for idx in sample_idx:
    sims = hybrid_similarity_row(idx)
    sims_no_self = sims.copy()
    sims_no_self[idx] = -np.inf

    top_k_idx = np.argpartition(sims_no_self, -K)[-K:]
    top_k_sims.append(sims_no_self[top_k_idx].mean())

    random_others = rng.choice([i for i in range(n_titles) if i != idx], size=K, replace=False)
    random_k_sims.append(sims[random_others].mean())

mean_top_k = float(np.mean(top_k_sims))
mean_random_k = float(np.mean(random_k_sims))
similarity_lift = mean_top_k / mean_random_k if mean_random_k > 0 else float('nan')

print(f"Mean similarity of top-{K} recommendations : {mean_top_k:.4f}")
print(f"Mean similarity of {K} random pairs         : {mean_random_k:.4f}")
print(f"Similarity lift (top-K / random)             : {similarity_lift:.2f}x")

Mean similarity of top-10 recommendations : 0.6555
Mean similarity of 10 random pairs         : 0.1293
Similarity lift (top-K / random)             : 5.07x


In [17]:
# ------------------------------------------------------------
# 2) Franchise self-retrieval rate (title-text-derived ground truth)
# ------------------------------------------------------------
import re

def franchise_key(title):
    """Strip common sequel/season/part markers so different entries of the
    same franchise collapse to the same key, e.g.
    'Boruto: Naruto Next Generations' and 'Boruto: Naruto Next Generations 2nd Season'."""
    t = str(title).lower()
    t = re.sub(r'\b(\d+(st|nd|rd|th)?\s*season)\b', '', t)
    t = re.sub(r'\bseason\s*\d+\b', '', t)
    t = re.sub(r'\bpart\s*\d+\b', '', t)
    t = re.sub(r'\b(ii|iii|iv|v|vi|vii|2nd|3rd|4th|5th)\b', '', t)
    t = re.sub(r'[^a-z0-9]+', ' ', t)
    return t.strip()

dt['franchise_key'] = dt['title'].apply(franchise_key)
franchise_counts = dt['franchise_key'].value_counts()
multi_entry_franchises = set(franchise_counts[franchise_counts >= 2].index) - {''}

candidates = dt[dt['franchise_key'].isin(multi_entry_franchises)].copy()
print(f"Titles with at least one detected same-franchise sibling: {len(candidates)} "
      f"(across {len(multi_entry_franchises)} franchises)")

EVAL_SAMPLE = min(300, len(candidates))
eval_idx = rng.choice(candidates.index.values, size=EVAL_SAMPLE, replace=False)

hits = 0
for idx in eval_idx:
    key = dt.loc[idx, 'franchise_key']
    sims = hybrid_similarity_row(idx)
    sims[idx] = -np.inf
    top10_idx = np.argpartition(sims, -10)[-10:]
    top10_keys = dt.iloc[top10_idx]['franchise_key'].values
    if key in top10_keys:
        hits += 1

franchise_recall_at_10 = hits / EVAL_SAMPLE
print(f"Franchise-recall@10 (n={EVAL_SAMPLE}): {franchise_recall_at_10:.1%}")

Titles with at least one detected same-franchise sibling: 1019 (across 416 franchises)
Franchise-recall@10 (n=300): 73.3%


In [18]:
from IPython.display import Markdown, display

rec_text = f"""
### Recommender evaluation summary

Using only the similarity scores the hybrid recommender already produces:

- **Similarity lift:** the system's own top-{K} recommendations score **{similarity_lift:.2f}x** higher, on
  average, than {K} randomly chosen, unrelated titles ({mean_top_k:.3f} vs. {mean_random_k:.3f} mean hybrid
  similarity). A lift well above 1.0 indicates the storyline + genre/theme embedding is capturing genuine
  narrative and content similarity rather than producing near-random rankings.
- **Franchise-recall@10:** of {EVAL_SAMPLE} titles with a detectable same-franchise sibling elsewhere in the
  dataset, the recommender's own top-10 results contained that sibling **{franchise_recall_at_10:.1%}** of the
  time. This is a real, independently verifiable relevance signal (derived purely from title text, not
  hand-labelled), and gives a concrete, defensible answer to "does the recommender actually work" using only
  the similarity scores the model computes internally.

Both checks avoid the need for any external ground-truth relevance labels, which is exactly the constraint
this recommender operates under (there is no "correct recommendation" label anywhere in the source data) —
consistent with the guidance that, for a similarity-based system, the similarity scores themselves are the
evaluation.
"""
display(Markdown(rec_text))


### Recommender evaluation summary

Using only the similarity scores the hybrid recommender already produces:

- **Similarity lift:** the system's own top-10 recommendations score **5.07x** higher, on
  average, than 10 randomly chosen, unrelated titles (0.656 vs. 0.129 mean hybrid
  similarity). A lift well above 1.0 indicates the storyline + genre/theme embedding is capturing genuine
  narrative and content similarity rather than producing near-random rankings.
- **Franchise-recall@10:** of 300 titles with a detectable same-franchise sibling elsewhere in the
  dataset, the recommender's own top-10 results contained that sibling **73.3%** of the
  time. This is a real, independently verifiable relevance signal (derived purely from title text, not
  hand-labelled), and gives a concrete, defensible answer to "does the recommender actually work" using only
  the similarity scores the model computes internally.

Both checks avoid the need for any external ground-truth relevance labels, which is exactly the constraint
this recommender operates under (there is no "correct recommendation" label anywhere in the source data) —
consistent with the guidance that, for a similarity-based system, the similarity scores themselves are the
evaluation.


### Limitations of this evaluation, stated honestly

- **Similarity lift** confirms the embedding space has real structure, but does not by itself confirm the *specific* 60/40 storyline/genre weighting is optimal — only that the overall hybrid score is informative relative to random chance.
- **Franchise-recall@10** is a conservative, narrow slice of "relevance" (it only credits an exact same-franchise hit); a recommender that surfaces a thematically very similar but unrelated title is not rewarded by this metric even if a human would consider it a good recommendation. It should be read as a lower bound on recommendation quality, not a complete picture.
- Neither check requires or uses labelled human relevance judgments, since none exist for this dataset — a genuine user study or expert relevance annotation remains the more rigorous (but far more time-consuming) gold-standard evaluation, and is noted as future work.